# 02 — Limpieza y Transformación de Datos
**Proyecto:** Dashboard Predictivo del Mercado Laboral Colombiano


## 1. Imports y configuración

In [ ]:
import pandas as pd
import numpy as np
import re
import os
from datetime import datetime

RAW_DIR = '../data/raw'
INTERMEDIATE_DIR = '../data/02_intermediate'
PRIMARY_DIR = '../data/03_primary'
os.makedirs(INTERMEDIATE_DIR, exist_ok=True)
os.makedirs(PRIMARY_DIR, exist_ok=True)
print('Directorios listos.')

## 2. Limpieza del dataset SENA
### 2.1 Problema: encoding especial en columnas con ñ

In [ ]:
df_sena_raw = pd.read_csv(f'{RAW_DIR}/sena/sena_inscritos.csv')
print('Columnas originales:')
print(df_sena_raw.columns.tolist())
print(f'\nRegistros: {len(df_sena_raw)}, Nulos por columna:')
print(df_sena_raw.isnull().sum())

In [ ]:
# Mapeo de columnas con encoding especial
COL_MAP = {
    'nombre_de_la_ocupaci_n': 'ocupacion',
    'n_mero_de_inscritos_2019': 'inscritos_2019',
    'n_mero_de_inscritos_2020': 'inscritos_2020'
}

# Renombrar solo columnas que existan
rename = {k: v for k, v in COL_MAP.items() if k in df_sena_raw.columns}
df_sena = df_sena_raw.rename(columns=rename)

# Convertir inscritos a numérico
for col in ['inscritos_2019', 'inscritos_2020']:
    if col in df_sena.columns:
        df_sena[col] = pd.to_numeric(df_sena[col], errors='coerce').fillna(0).astype(int)

# Limpiar texto de ocupación
df_sena['ocupacion'] = df_sena['ocupacion'].str.strip().str.title()

print(f'Columnas limpias: {df_sena.columns.tolist()}')
df_sena.head()

In [ ]:
# Guardar versión limpia
df_sena.to_csv(f'{INTERMEDIATE_DIR}/sena_limpio.csv', index=False)
print(f'SENA limpio guardado: {len(df_sena)} registros')

## 3. Limpieza de la serie temporal DANE

In [ ]:
df_serie = pd.read_csv('../data/processed/serie_temporal_td.csv', parse_dates=['fecha'])
print(f'Registros: {len(df_serie)}')
print(f'Nulos: {df_serie.isnull().sum().to_dict()}')

# Validar rangos
VALIDACIONES = {
    'tasa_desocupacion': (5, 25),
    'tasa_ocupacion': (45, 70),
    'tasa_global_participacion': (55, 80)
}

for col, (vmin, vmax) in VALIDACIONES.items():
    fuera = df_serie[(df_serie[col] < vmin) | (df_serie[col] > vmax)]
    if len(fuera) > 0:
        print(f'⚠️  {col}: {len(fuera)} valores fuera de rango [{vmin}, {vmax}]')
    else:
        print(f'✅ {col}: todos los valores en rango [{vmin}, {vmax}]')

In [ ]:
# Calcular variación anual si no existe
if 'variacion_anual_td' not in df_serie.columns:
    df_serie = df_serie.sort_values('fecha').reset_index(drop=True)
    df_serie['variacion_anual_td'] = df_serie['tasa_desocupacion'].diff(12)
    print('Variación anual calculada.')

# Guardar versión intermedia
df_serie.to_csv(f'{INTERMEDIATE_DIR}/serie_temporal_limpia.csv', index=False)
print(f'Serie limpia guardada: {len(df_serie)} registros')
df_serie.head()

## 4. Integración y consolidado primario

In [ ]:
# Agregar columnas de período para joins
df_serie['anio'] = df_serie['fecha'].dt.year
df_serie['mes'] = df_serie['fecha'].dt.month
df_serie['trimestre'] = df_serie['fecha'].dt.quarter

# Agregar total inscritos SENA por año para cruce con serie
total_sena = {
    2019: df_sena['inscritos_2019'].sum(),
    2020: df_sena['inscritos_2020'].sum()
}
df_serie['total_inscritos_sena'] = df_serie['anio'].map(total_sena)

print('Dataset consolidado:')
print(df_serie[['fecha','tasa_desocupacion','variacion_anual_td','total_inscritos_sena']].head(10))

df_serie.to_csv(f'{PRIMARY_DIR}/dataset_consolidado.csv', index=False)
print(f'\nDataset primario guardado.')

## 5. Resumen de transformaciones

| Fuente | Problema | Solución |
|---|---|---|
| SENA CSV | Encoding ñ → _n_ | Mapeo explícito de columnas |
| SENA CSV | Inscritos como string | Conversión a int, NaN → 0 |
| Serie DANE | Sin variación anual | Calculada con diff(12) |
| Consolidado | Sin período estructurado | Añadidas columnas anio/mes/trimestre |